Connect to the same database `faq.db` and check how many documents are in the index

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

sqlite_index.count()

83

In [2]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I missed the first homework - can I still get a certificate?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

In [3]:
from dotenv import load_dotenv
load_dotenv()
import os
from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [4]:
from rag_helper import RAGBase

assistant = RAGBase(sqlite_index, client)

In [5]:
assistant.rag('I just discovered the course, can I still join?')

'Yes—you can still join the course. Just keep in mind that if you’d like to earn a certificate, you’ll need to submit your project before the submission deadline closes.'

The answer should be similar to what we got with minsearch. But now the data comes from a persistent index

- `ingest.py` handles data loading and indexing
- `rag_helper.py` handles the RAG pipeline
- The notebooks wire them together

With minsearch (single process):

```text
Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
```

With sqlitesearch (two processes):

```text
Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready
```

In [6]:
# When you're done, close the database connection
sqlite_index.close()